# Tiny GPT Pretraining on Shakespeare

Companion notebook for the [Pretraining Objectives](https://codemia.io/courses/llm_inside_and_out/pretraining_objectives) lesson on Codemia.

**What you will build**: a character-level GPT with 6 transformer layers, 4 heads, and 128-dim embeddings, trained from scratch on tiny Shakespeare for ~5000 steps. You will watch the loss curve fall, sample at several checkpoints, and see the model go from gibberish to plausible Elizabethan-looking text.

**Karpathy companion**: [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY). This notebook is a condensed version of that video's final model. Adapted in part from karpathy/nanoGPT (MIT).

**Time**: ~20-30 minutes training on T4 GPU

**GPU required**: yes

## Setup

In [ ]:
%pip install -q torch numpy matplotlib requests

In [ ]:
import math, time, requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

## Concept recap

Pretraining a language model is just next-token prediction. Given a sequence `(x_1, ..., x_T)` we ask the model to predict `x_{t+1}` from `(x_1, ..., x_t)` for every `t`, and minimize the sum of cross-entropy losses. That's the whole objective. The magic is that this simple loss, scaled to billions of tokens, produces models that can translate, reason, and write code.

At tiny-Shakespeare scale (~1M characters) we won't get a coherent model, but we will see the three signature moments of pretraining:

1. Random outputs at step 0 (pure noise from the softmax over 65 chars)
2. Memorizing character frequencies and common bigrams by step ~200
3. Picking up Shakespeare-style structure (line breaks, character names in caps, verse cadence) by step ~3000

## Data: tiny Shakespeare

In [ ]:
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
text = requests.get(url).text
print(f"{len(text):,} characters")
print("---- first 150 chars ----")
print(text[:150])

In [ ]:
# Build a char-level vocabulary. 65 unique chars including newlines and uppercase.
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

def encode(s): return [stoi[c] for c in s]
def decode(ids): return ''.join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"vocab_size={vocab_size}, train={len(train_data):,}, val={len(val_data):,}")

In [ ]:
# Hyperparameters. These match the config in the "Let's build GPT" video
# closely enough that you should see similar output quality after 5k steps.
batch_size = 64
block_size = 128
n_embd = 128
n_head = 4
n_layer = 6
dropout = 0.1
learning_rate = 3e-4
max_steps = 5000
eval_every = 500
eval_iters = 20

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

## Implementation: the GPT model (nanoGPT-flavored)

In [ ]:
class CausalSelfAttention(nn.Module):
    # Same layer you built in transformer_block.ipynb, with dropout and the qkv fused
    # into one linear for speed.
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head, self.n_embd = n_head, n_embd
        # Use F.scaled_dot_product_attention which dispatches to flash on GPU.
        # This is the nanoGPT / build-nanogpt modern style.

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        d_head = C // self.n_head
        q = q.view(B, T, self.n_head, d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, d_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

In [ ]:
class MLP(nn.Module):
    # 4x up-projection, GELU, 4x down-projection. The standard GPT FFN.
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(F.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        # Residual stream with pre-LN. Same structure as transformer_block.ipynb.
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        # Token and position embeddings. Learned absolute positions (nanoGPT style).
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        # Weight tying: the unembedding head shares weights with the input embedding table.
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        # Small-std init so the residual stream stays well-scaled through many layers.
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                torch.nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            # Always condition on the last block_size tokens so we never exceed the position table.
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
model = GPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params/1e6:.2f}M parameters")
print(f"  tok_emb: {model.tok_emb.weight.numel():,}")
print(f"  pos_emb: {model.pos_emb.weight.numel():,}")
print(f"  blocks:  {sum(p.numel() for p in model.blocks.parameters()):,}")

## Sample BEFORE training (this is the baseline: pure noise)

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
sample = decode(model.generate(context, 200)[0].tolist())
print("---- untrained sample ----")
print(sample)

## Training loop

In [ ]:
@torch.no_grad()
def estimate_loss():
    # Mean over a handful of batches per split, for a less noisy loss estimate than a single batch.
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

train_curve, val_curve = [], []
checkpoint_samples = {}
t0 = time.time()

for step in range(max_steps + 1):
    if step % eval_every == 0:
        losses = estimate_loss()
        train_curve.append((step, losses['train']))
        val_curve.append((step, losses['val']))
        # Quick 200-char sample at this checkpoint so we can see quality improve over time.
        ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
        checkpoint_samples[step] = decode(model.generate(ctx, 200)[0].tolist())
        print(f"step {step:5d}  train {losses['train']:.3f}  val {losses['val']:.3f}  "
              f"elapsed {time.time()-t0:5.1f}s")

    if step == max_steps:
        break
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"done in {time.time()-t0:.1f}s")

## Loss curve

In [ ]:
tr_x, tr_y = zip(*train_curve)
va_x, va_y = zip(*val_curve)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tr_x, tr_y, 'o-', label='train')
ax.plot(va_x, va_y, 's-', label='val')
ax.set_xlabel('step')
ax.set_ylabel('cross-entropy loss')
ax.set_title('Tiny-GPT pretraining on Shakespeare')
ax.legend()
plt.tight_layout()
plt.show()

import math as _m
print(f"final val loss: {val_curve[-1][1]:.3f}  -> perplexity {_m.exp(val_curve[-1][1]):.2f}")

## Sample evolution over training

Look at these three snapshots. Step 0 is gibberish. Around step 1000 the model has learned character frequencies and starts making word-like strings. By step 5000 you can see proper line breaks, capitalized character names, and the beginnings of verse-like line length.

In [ ]:
for step in [0, 500, 2500, 5000]:
    if step not in checkpoint_samples:
        continue
    print(f"======= step {step} =======")
    print(checkpoint_samples[step])
    print()

## Variation 1: sample at different temperatures

Temperature divides the logits before softmax. T=1 is the base distribution. T<1 makes the model more confident (collapses toward greedy). T>1 flattens the distribution (more chaotic).

In [ ]:
ctx = torch.tensor([encode("ROMEO:")], device=device)
for temp in [0.3, 0.8, 1.5]:
    out = model.generate(ctx, 200, temperature=temp)[0].tolist()
    print(f"------- temperature = {temp} -------")
    print(decode(out))
    print()

## Variation 2: a prompt the model has definitely not seen

Give it a made-up Shakespearean-style opening and see what it continues with. This probes generalization — how well did it learn the *form* vs the literal training text?

In [ ]:
prompt = "GHOST of the FORGOTTEN KING:\n"
ctx = torch.tensor([encode(prompt)], device=device)
out = model.generate(ctx, 300, temperature=0.8)[0].tolist()
print(decode(out))

## What to try next

- Bump `n_layer` to 8 and `n_embd` to 256 and train for 10k steps. Loss should drop noticeably, sampled text should look more coherent.
- Swap the BPE tokenizer from `tiktoken` (GPT-2's encoder) in place of the char-level one. You'll need ~2x fewer training steps per unit of perceived quality.
- Train on a different corpus (e.g., Python source code or a paper abstract dataset) and see which style the model picks up.
- Profile one training step with `torch.profiler` and identify which op is the bottleneck. On T4 it's probably the matmul in the FFN.
- Continue the training from this checkpoint in the `finetuning/full_finetune_catastrophic_forgetting.ipynb` notebook to see how fine-tuning changes its behavior.